In [ ]:
%pip install foundry-local-sdk
%pip install openai
%pip install agent-framework --pre
%pip install pywin32

In [1]:
from foundry_local import FoundryLocalManager
from agent_framework.openai import OpenAIChatClient
import openai

In [2]:
# Initialize and optionally bootstrap with a model
manager = FoundryLocalManager(
    alias_or_model_id="phi-4-mini",
    bootstrap=True
)

In [3]:
print(f'Service endpoint: {manager.endpoint}')
print(f'Service Api Key: {manager.api_key}')

Service endpoint: http://127.0.0.1:58728/v1
Service Api Key: OPENAI_API_KEY


In [4]:
# List loaded models
loaded = manager.list_loaded_models()
print(f"Models running in the service: {loaded}")

Models running in the service: [FoundryModelInfo(alias=phi-4-mini, id=Phi-4-mini-instruct-cuda-gpu:5, execution_provider=CUDAExecutionProvider, device_type=GPU, file_size=3686 MB, license=MIT)]


In [5]:
# Use OpenAI SDK
# Configure the client to use the local Foundry service
client = openai.OpenAI(
    base_url=manager.endpoint,
    api_key=manager.api_key  # API key is not required for local usage
)


In [6]:
stream = client.chat.completions.create(
    model=manager.get_model_info('phi-4-mini').id,
    messages=[{"role": "user", "content": "Why is the sky blue?"}],
    stream=True
)

In [ ]:
for chunk in stream:
    if chunk.choices[0].delta.content is not None:
        print(chunk.choices[0].delta.content, end="", flush=True)

In [8]:
from agent_framework import Agent
from agent_framework.openai import OpenAIChatClient

In [13]:
client_openai = OpenAIChatClient(
    model_id=manager.get_model_info('phi-4-mini').id,
    base_url=manager.endpoint,
    api_key=manager.api_key
)

In [14]:
agent = Agent(
    client=client_openai,
    instructions="You are a fact agent"
)

In [15]:
response = await agent.run('Why is the sky blue?')

In [16]:
response.text

"The sky appears blue due to Rayleigh scattering, which is the scattering of sunlight by the molecules and small particles in the Earth's atmosphere. Sunlight is made up of different colors, each with different wavelengths. Blue light has a shorter wavelength and is scattered in all directions by the gases and particles in the atmosphere much more than other colors with longer wavelengths. This scattering causes the blue color to be seen when we look up at the sky during the day. At sunrise and sunset, the sky can appear red or orange because the sunlight has to pass through a greater thickness of the Earth's atmosphere, which scatters the shorter wavelengths and allows the longer wavelengths to reach our eyes."

In [ ]:
# Unload a model
manager.unload_model(loaded[0].alias)